# A3: Predicting Car Price III (Task 1 and Task 2 Only)

**Course:** AT82.03 Machine Learning  
**Student:** Puritaporn Thongsuvan  
**Student ID:** st127397

This file continues from the A2 car-price. The deployment part of Task 3 is skipped, following my discussion with professor.  

## Data Preparation from A1/A2

I reuse the same cleaning logic from the A2 notebook so that Task 1 and Task 2 use the preprocessed version of the dataset, as required by the assignment.  


**Step 1 - Import Libraries**

In [1]:
# Import basic packages, plotting tools, and scikit-learn utilities used in this notebook.

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.validation import check_is_fitted

try:
    from scipy import sparse
except ImportError:
    sparse = None

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

**Step 2 - Load Dataset**

The notebook expects `Cars.csv` to be in the same folder as this notebook, the same as the previous A1/A2 work.  


In [2]:
# Load the original car-price dataset and inspect the first rows.

df = pd.read_csv("Cars.csv")

display(df.head())
print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


Shape: (8128, 13)
Duplicate rows: 1202

Missing values:


,missing
torque,222
seats,221
engine,221
mileage,221
max_power,215
fuel,0
km_driven,0
selling_price,0
year,0
name,0


**Step 3 - Clean Data**

Cleaning decisions reused from A2:

- Convert `owner` from text order to numeric order.
- Remove CNG and LPG because their mileage unit is different.
- Convert `mileage`, `engine`, and `max_power` to numeric values.
- Create `brand` from the first word of `name`.
- Remove `torque` and Test Drive Cars.

In [3]:
# Encode owner categories and remove fuel types with a different mileage unit.

owner_order = {
    "First Owner": 1,
    "Second Owner": 2,
    "Third Owner": 3,
    "Fourth & Above Owner": 4,
    "Test Drive Car": 5,
}

df["owner"] = df["owner"].map(owner_order)
df = df[~df["fuel"].isin(["CNG", "LPG"])].copy()
df = df.reset_index(drop=True)

In [4]:
# Remove unit text and convert mileage, engine, and max_power to floats.

df["mileage"] = df["mileage"].astype(str).str.split(" ").str[0].astype(float)
df["engine"] = df["engine"].astype(str).str.split(" ").str[0].astype(float)
df["max_power"] = df["max_power"].astype(str).str.split(" ").str[0].astype(float)

In [5]:
# Create the brand column, remove unused data, and keep the final columns.

df["brand"] = df["name"].str.split().str[0]
df = df.drop(columns=["torque"]).copy()
df = df[df["owner"] != 5].copy()

selected_columns = [
    "brand", "year", "selling_price", "km_driven", "fuel",
    "seller_type", "transmission", "owner", "mileage", "engine",
    "max_power", "seats",
]
df = df[selected_columns].copy()

print("Rows after cleaning:", len(df))
display(df.head())

Rows after cleaning: 8028


,brand,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats
0,Maruti,2014,450000,145500,Diesel,Individual,Manual,1,23.40,1248.0,74.00,5.0
1,Skoda,2014,370000,120000,Diesel,Individual,Manual,2,21.14,1498.0,103.52,5.0
2,Honda,2006,158000,140000,Petrol,Individual,Manual,3,17.70,1497.0,78.00,5.0
3,Hyundai,2010,225000,127000,Diesel,Individual,Manual,1,23.00,1396.0,90.00,5.0
4,Maruti,2007,130000,120000,Petrol,Individual,Manual,1,16.10,1298.0,88.20,5.0


# Task 1: Classification with Custom Logistic Regression

**Step 4 - Convert Selling Price to 4 Classes / แปลงราคาเป็น 4 Classes**

I use quantile-based buckets with `pd.qcut()` so that each class has a similar number of samples. This helps prevent one class from becoming too small because car prices are usually skewed.  

In [6]:
# Convert the continuous selling price into four ordered classes: 0, 1, 2, and 3.

price_labels = [0, 1, 2, 3]
df["price_class"] = pd.qcut(
    df["selling_price"],
    q=4,
    labels=price_labels,
    duplicates="drop",
).astype(int)

print("Price-class counts:")
display(df["price_class"].value_counts().sort_index().to_frame("count"))

print("\nSelling-price range in each class:")
display(
    df.groupby("price_class")["selling_price"]
    .agg(["min", "max", "count"])
    .rename(columns={"min": "min_price", "max": "max_price"})
)

Price-class counts:


,count
price_class,
0,2050
1,2044
2,1943
3,1991



Selling-price range in each class:


,min_price,max_price,count
price_class,,,
0,29999,260000,2050
1,261000,450000,2044
2,451000,680000,1943
3,681000,10000000,1991


**Step 5 - Split Features and Labels**

The features are all columns except `selling_price` and `price_class`. The label is `price_class`. I use `stratify=y` so that train and test keep similar class proportions.  

In [7]:
# Create X and y for classification and split the data into training and test sets.

X = df.drop(columns=["selling_price", "price_class"])
y = df["price_class"]

num_cols = ["year", "km_driven", "owner", "mileage", "engine", "max_power", "seats"]
cat_cols = ["brand", "fuel", "seller_type", "transmission"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train class counts:")
display(y_train.value_counts().sort_index().to_frame("train_count"))
print("y_test class counts:")
display(y_test.value_counts().sort_index().to_frame("test_count"))

X_train shape: (5619, 11)
X_test shape : (2409, 11)
y_train class counts:


,train_count
price_class,
0,1435
1,1431
2,1360
3,1393


y_test class counts:


,test_count
price_class,
0,615
1,613
2,583
3,598


## Step 6 - Build Preprocessor

Missing numerical values are filled with the median, numerical variables are scaled, and categorical variables are one-hot encoded. The preprocessor is fitted only on training data to avoid data leakage.

In [8]:
# Create OneHotEncoder in a way that works for both old and new scikit-learn versions.

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


# Build the preprocessing pipeline for numeric and categorical columns.

def build_preprocessor():
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, num_cols),
            ("cat", categorical_pipeline, cat_cols),
        ]
    )

**Step 7 - Implement Multinomial Logistic Regression**

This class uses softmax for multi-class classification. It supports normal Logistic Regression and Ridge/L2 Logistic Regression.  

In [9]:
# Custom multinomial Logistic Regression implemented from scratch.

class LogisticRegression(BaseEstimator, ClassifierMixin):
    
    # Multinomial Logistic Regression with optional Ridge/L2 penalty.

    def __init__(
        self,
        learning_rate=0.05,
        max_iter=2000,
        penalty=None,
        l2_lambda=0.0,
        tol=1e-7,
        fit_intercept=True,
        random_state=42,
        verbose=False,
    ):
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.penalty = penalty
        self.l2_lambda = l2_lambda
        self.tol = tol
        self.fit_intercept = fit_intercept
        self.random_state = random_state
        self.verbose = verbose

    def _to_numpy(self, X):
        # Convert pandas or sparse matrix input to a NumPy array.
        if sparse is not None and sparse.issparse(X):
            X = X.toarray()
        elif hasattr(X, "to_numpy"):
            X = X.to_numpy()
        return np.asarray(X, dtype=float)

    def _add_intercept(self, X):
        # Add a column of ones for the intercept term if requested.
        if not self.fit_intercept:
            return X
        intercept = np.ones((X.shape[0], 1))
        return np.c_[intercept, X]

    def _softmax(self, z):
        # Stable softmax prevents overflow by subtracting the row maximum.
        z_stable = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z_stable)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def _one_hot(self, y):
        # Convert class labels into one-hot encoded rows.
        return (y.reshape(-1, 1) == self.classes_.reshape(1, -1)).astype(float)

    def _ridge_is_used(self):
        # Accept both 'l2' and 'ridge' as names for the same penalty.
        return self.penalty in ["l2", "ridge"] and self.l2_lambda > 0

    def fit(self, X, y):
        # Fit the model by gradient descent.
        X = self._to_numpy(X)
        y = np.asarray(y)

        self.classes_ = np.unique(y)
        y_encoded = self._one_hot(y)
        X_design = self._add_intercept(X)

        n_samples, n_features = X_design.shape
        n_classes = len(self.classes_)

        rng = np.random.default_rng(self.random_state)
        self.theta_ = rng.normal(loc=0.0, scale=0.01, size=(n_features, n_classes))
        self.loss_history_ = []

        previous_loss = np.inf
        eps = 1e-15

        for iteration in range(self.max_iter):
            logits = X_design @ self.theta_
            probabilities = self._softmax(logits)

            # Cross-entropy loss averaged by sample count for numerical stability.
            data_loss = -np.mean(np.sum(y_encoded * np.log(probabilities + eps), axis=1))

            # Ridge/L2 does not regularize the intercept row.
            if self._ridge_is_used():
                ridge_loss = (self.l2_lambda / n_samples) * np.sum(self.theta_[1:] ** 2)
            else:
                ridge_loss = 0.0

            loss = data_loss + ridge_loss
            self.loss_history_.append(loss)

            # Gradient of softmax cross-entropy.
            gradient = (X_design.T @ (probabilities - y_encoded)) / n_samples

            if self._ridge_is_used():
                ridge_gradient = (2 * self.l2_lambda / n_samples) * self.theta_
                ridge_gradient[0, :] = 0.0
                gradient += ridge_gradient

            self.theta_ -= self.learning_rate * gradient

            if self.verbose and iteration % 200 == 0:
                print(f"Iteration {iteration:4d} | loss = {loss:.6f}")

            if abs(previous_loss - loss) < self.tol:
                break
            previous_loss = loss

        self.n_iter_ = iteration + 1
        return self

    def predict_proba(self, X):
        # Return class probabilities for each sample.
        check_is_fitted(self, ["theta_", "classes_"])
        X = self._to_numpy(X)
        X_design = self._add_intercept(X)
        return self._softmax(X_design @ self.theta_)

    def predict(self, X):
        # Predict the class with the highest probability.
        probabilities = self.predict_proba(X)
        class_indices = np.argmax(probabilities, axis=1)
        return self.classes_[class_indices]

    def score(self, X, y):
        # Make the class compatible with scikit-learn style estimators.
        return accuracy(y, self.predict(X))

**Step 8 - Implement Classification Metrics from Scratch**

These functions reproduce the main values in scikit-learn's `classification_report`: accuracy, precision, recall, f1-score, macro average, weighted average, and support.  

In [10]:
# Safely divide two numbers; return 0 if the denominator is zero.

def safe_divide(numerator, denominator):
    return 0.0 if denominator == 0 else numerator / denominator


# Accuracy = correct predictions / all predictions.

def accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(y_true == y_pred)


# Calculate TP, FP, FN, and TN for one selected class.

def confusion_values_for_class(y_true, y_pred, class_label):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    tp = np.sum((y_true == class_label) & (y_pred == class_label))
    fp = np.sum((y_true != class_label) & (y_pred == class_label))
    fn = np.sum((y_true == class_label) & (y_pred != class_label))
    tn = np.sum((y_true != class_label) & (y_pred != class_label))
    return tp, fp, fn, tn


# Precision for class c = TP_c / (TP_c + FP_c).

def precision(y_true, y_pred, class_label):
    tp, fp, fn, tn = confusion_values_for_class(y_true, y_pred, class_label)
    return safe_divide(tp, tp + fp)


# Recall for class c = TP_c / (TP_c + FN_c).

def recall(y_true, y_pred, class_label):
    tp, fp, fn, tn = confusion_values_for_class(y_true, y_pred, class_label)
    return safe_divide(tp, tp + fn)


# F1-score is the harmonic mean of precision and recall.

def f1_score(y_true, y_pred, class_label):
    p = precision(y_true, y_pred, class_label)
    r = recall(y_true, y_pred, class_label)
    return safe_divide(2 * p * r, p + r)


# Support means the number of true samples in each class.

def support(y_true, class_label):
    y_true = np.asarray(y_true)
    return np.sum(y_true == class_label)


# Macro average gives every class equal weight.

def macro_precision(y_true, y_pred, labels):
    return np.mean([precision(y_true, y_pred, label) for label in labels])


def macro_recall(y_true, y_pred, labels):
    return np.mean([recall(y_true, y_pred, label) for label in labels])


def macro_f1(y_true, y_pred, labels):
    return np.mean([f1_score(y_true, y_pred, label) for label in labels])


# Weighted average gives more weight to classes with more samples.

def weighted_precision(y_true, y_pred, labels):
    total_support = sum(support(y_true, label) for label in labels)
    return safe_divide(
        sum(support(y_true, label) * precision(y_true, y_pred, label) for label in labels),
        total_support,
    )


def weighted_recall(y_true, y_pred, labels):
    total_support = sum(support(y_true, label) for label in labels)
    return safe_divide(
        sum(support(y_true, label) * recall(y_true, y_pred, label) for label in labels),
        total_support,
    )


def weighted_f1(y_true, y_pred, labels):
    total_support = sum(support(y_true, label) for label in labels)
    return safe_divide(
        sum(support(y_true, label) * f1_score(y_true, y_pred, label) for label in labels),
        total_support,
    )


# Create a table similar to sklearn.metrics.classification_report.

def custom_classification_report(y_true, y_pred, labels=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if labels is None:
        labels = np.unique(np.r_[y_true, y_pred])

    rows = []
    for label in labels:
        rows.append(
            {
                "class": label,
                "precision": precision(y_true, y_pred, label),
                "recall": recall(y_true, y_pred, label),
                "f1-score": f1_score(y_true, y_pred, label),
                "support": support(y_true, label),
            }
        )

    report = pd.DataFrame(rows).set_index("class")
    report.loc["accuracy", ["precision", "recall", "f1-score", "support"]] = [
        np.nan,
        np.nan,
        accuracy(y_true, y_pred),
        len(y_true),
    ]
    report.loc["macro avg"] = [
        macro_precision(y_true, y_pred, labels),
        macro_recall(y_true, y_pred, labels),
        macro_f1(y_true, y_pred, labels),
        len(y_true),
    ]
    report.loc["weighted avg"] = [
        weighted_precision(y_true, y_pred, labels),
        weighted_recall(y_true, y_pred, labels),
        weighted_f1(y_true, y_pred, labels),
        len(y_true),
    ]

    return report

**Step 9 - Check Metrics with Mockup Data**

The assignment asks us to compare our implementation with scikit-learn. I first test it on small mockup data where checking by hand is easier.  

In [11]:
# Use simple mockup labels to compare custom metrics with sklearn.

mock_y_true = np.array([0, 0, 1, 1, 2, 2, 3, 3, 3])
mock_y_pred = np.array([0, 1, 1, 1, 2, 0, 3, 2, 3])
mock_labels = np.array([0, 1, 2, 3])

print("Custom classification report:")
display(custom_classification_report(mock_y_true, mock_y_pred, labels=mock_labels).round(4))

print("sklearn classification_report:")
print(classification_report(mock_y_true, mock_y_pred, labels=mock_labels, zero_division=0))

Custom classification report:


,precision,recall,f1-score,support
class,,,,
0,0.5000,0.5000,0.5000,2.0
1,0.6667,1.0000,0.8000,2.0
2,0.5000,0.5000,0.5000,2.0
3,1.0000,0.6667,0.8000,3.0
accuracy,NaN,NaN,0.6667,9.0
macro avg,0.6667,0.6667,0.6500,9.0
weighted avg,0.7037,0.6667,0.6667,9.0


sklearn classification_report:
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.67      1.00      0.80         2
           2       0.50      0.50      0.50         2
           3       1.00      0.67      0.80         3

    accuracy                           0.67         9
   macro avg       0.67      0.67      0.65         9
weighted avg       0.70      0.67      0.67         9



# Task 2: Ridge Logistic Regression

```text
J(theta) = - sum y(i) log(h(i)) + lambda * sum theta_j^2
```
I use the averaged loss version for stable training, and I do not regularize the intercept term.

**Step 10 - Train Normal Logistic Regression**

In [12]:
# Build a pipeline with preprocessing and the custom Logistic Regression without Ridge penalty.

normal_logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        (
            "classifier",
            LogisticRegression(
                learning_rate=0.05,
                max_iter=2000,
                penalty=None,
                l2_lambda=0.0,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

normal_logistic_pipeline.fit(X_train, y_train)
normal_pred = normal_logistic_pipeline.predict(X_test)

print("Normal Logistic Regression")
print("Accuracy:", accuracy(y_test, normal_pred))
print("Iterations:", normal_logistic_pipeline.named_steps["classifier"].n_iter_)
display(custom_classification_report(y_test, normal_pred, labels=price_labels).round(4))

Normal Logistic Regression
Accuracy: 0.7459526774595268
Iterations: 2000


,precision,recall,f1-score,support
class,,,,
0,0.8798,0.8569,0.8682,615.0
1,0.6942,0.6852,0.6897,613.0
2,0.6180,0.6244,0.6212,583.0
3,0.7890,0.8127,0.8007,598.0
accuracy,NaN,NaN,0.7460,2409.0
macro avg,0.7452,0.7448,0.7449,2409.0
weighted avg,0.7467,0.7460,0.7462,2409.0


**Step 11 - Train Ridge/L2 Logistic Regression**

In [13]:
# Build a pipeline with preprocessing and the custom Logistic Regression with Ridge/L2 penalty.

ridge_logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        (
            "classifier",
            LogisticRegression(
                learning_rate=0.05,
                max_iter=2000,
                penalty="l2",
                l2_lambda=1.0,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

ridge_logistic_pipeline.fit(X_train, y_train)
ridge_pred = ridge_logistic_pipeline.predict(X_test)

print("Ridge/L2 Logistic Regression")
print("Accuracy:", accuracy(y_test, ridge_pred))
print("Iterations:", ridge_logistic_pipeline.named_steps["classifier"].n_iter_)
display(custom_classification_report(y_test, ridge_pred, labels=price_labels).round(4))

Ridge/L2 Logistic Regression
Accuracy: 0.746782897467829
Iterations: 2000


,precision,recall,f1-score,support
class,,,,
0,0.8798,0.8569,0.8682,615.0
1,0.6942,0.6852,0.6897,613.0
2,0.6212,0.6244,0.6228,583.0
3,0.7884,0.8161,0.8020,598.0
accuracy,NaN,NaN,0.7468,2409.0
macro avg,0.7459,0.7456,0.7456,2409.0
weighted avg,0.7473,0.7468,0.7469,2409.0


**Step 12 - Compare Normal vs Ridge**

In [14]:
# Compare normal Logistic Regression and Ridge Logistic Regression using the same metrics

comparison_df = pd.DataFrame(
    [
        {
            "model": "Normal Logistic Regression",
            "accuracy": accuracy(y_test, normal_pred),
            "macro_precision": macro_precision(y_test, normal_pred, price_labels),
            "macro_recall": macro_recall(y_test, normal_pred, price_labels),
            "macro_f1": macro_f1(y_test, normal_pred, price_labels),
            "weighted_precision": weighted_precision(y_test, normal_pred, price_labels),
            "weighted_recall": weighted_recall(y_test, normal_pred, price_labels),
            "weighted_f1": weighted_f1(y_test, normal_pred, price_labels),
        },
        {
            "model": "Ridge/L2 Logistic Regression",
            "accuracy": accuracy(y_test, ridge_pred),
            "macro_precision": macro_precision(y_test, ridge_pred, price_labels),
            "macro_recall": macro_recall(y_test, ridge_pred, price_labels),
            "macro_f1": macro_f1(y_test, ridge_pred, price_labels),
            "weighted_precision": weighted_precision(y_test, ridge_pred, price_labels),
            "weighted_recall": weighted_recall(y_test, ridge_pred, price_labels),
            "weighted_f1": weighted_f1(y_test, ridge_pred, price_labels),
        },
    ]
)

display(comparison_df.round(4))

,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1
0,Normal Logistic Regression,0.7460,0.7452,0.7448,0.7449,0.7467,0.7460,0.7462
1,Ridge/L2 Logistic Regression,0.7468,0.7459,0.7456,0.7456,0.7473,0.7468,0.7469


**Step 13 - Compare Custom Report with scikit-learn**

This confirms that the custom precision, recall, f1-score, support, macro average, and weighted average are close to scikit-learn's implementation.

In [15]:
# Compare custom classification report and sklearn classification_report for the Ridge model.

print("Custom report for Ridge/L2 Logistic Regression:")
display(custom_classification_report(y_test, ridge_pred, labels=price_labels).round(4))

print("sklearn classification_report for Ridge/L2 Logistic Regression:")
print(classification_report(y_test, ridge_pred, labels=price_labels, zero_division=0))

Custom report for Ridge/L2 Logistic Regression:


,precision,recall,f1-score,support
class,,,,
0,0.8798,0.8569,0.8682,615.0
1,0.6942,0.6852,0.6897,613.0
2,0.6212,0.6244,0.6228,583.0
3,0.7884,0.8161,0.8020,598.0
accuracy,NaN,NaN,0.7468,2409.0
macro avg,0.7459,0.7456,0.7456,2409.0
weighted avg,0.7473,0.7468,0.7469,2409.0


sklearn classification_report for Ridge/L2 Logistic Regression:
              precision    recall  f1-score   support

           0       0.88      0.86      0.87       615
           1       0.69      0.69      0.69       613
           2       0.62      0.62      0.62       583
           3       0.79      0.82      0.80       598

    accuracy                           0.75      2409
   macro avg       0.75      0.75      0.75      2409
weighted avg       0.75      0.75      0.75      2409



**Step 14 - Answer Brief Question: What does support in the classification report means?**

In a classification report, **support** means the number of true samples that belong to each class. For example, if class `2` has support `600`, it means there are 600 actual test samples whose true label is class `2`.